## Feature Engineering — Fraudulent Transaction Detection

This notebook picks up from EDA_Data.csv, and turns the patterns found there into a modelling-ready feature set: engineering threshold-based risk features directly motivated by the EDA findings, removing columns that shouldn't feed a model (identifiers, temporary helper columns, and leakage-risk fields), and splitting the remaining columns into categorical and numerical groups ahead of preprocessing.

In [ ]:
# importing libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV


In [2]:
Cleaned_cust_trans_data = pd.read_csv(r'C:/Users/oluwa/OneDrive/Desktop/Fraudulent_transaction_Detection_for_a_FinTech_Company/Finlora_Dataset/artifacts/EDA_Data.csv')

### Threshold-Based Feature Engineering

Each item below is a direct, deliberate translation of a specific EDA finding into a binary flag rather than an arbitrary guess — the EDA notebook found that several of the strongest fraud relationships were threshold/step-function effects rather than smooth trends, which is precisely the sort of pattern a linear model (such as Logistic Regression) cannot discover on its own from a raw continuous value, but which it can immediately pick up once it has been handed as an explicit flag:

-**late_night_hours** (hour 3–8): fraud rate peaked at 21.5% in this window compared to a 4.5–8.1% range for the rest of the day.

-**high_ip_risk** (IP risk score > 0.8): fraud rate jumped from ~1.5–2.1% below 0.6 to 62.9% above 0.8.

-**low_device_trust** (device trust score < 0.5): fraud rate was 85.0% below 0.3 and 11.4% for 0.3–0.5 vs under 3.3% for every band above 0.5.

-**new_account** (30–90 days) / **very_new_account** (under 30): fraud rate was 44.5% and 37.0%, vs under 2.5% for accounts over 90 days old.

-**velocity_spike** (1-hour transaction velocity ≥ 3): fraud rate jumped from near-0% at 0–2 transactions to ~83–85% at 3 and above.

-**amount_high** (amount > $500): chosen as the threshold because $500 is where the fraud rate visibly starts to climb in the EDA data — rising from 11.4% ($250–500) to 30.4% ($500–1k) — marking the point where a transaction moves from "everyday small" to "worth a fraudster's attention." This threshold is deliberately kept simple rather than trying to encode the full $1k–5k peak directly: a single binary split gives the model an easy, low-noise signal for "non-trivial amount," while the finer-grained pattern (including the peak at $1k–5k and the pullback above $5k) is preserved separately through the  amount_usd column already carried in the feature set.

In [4]:
# Creating a threshold based features from the following risk_signals

Cleaned_cust_trans_data['timestamp'] = pd.to_datetime(Cleaned_cust_trans_data['timestamp'])
Cleaned_cust_trans_data['late_night_hours'] = ((Cleaned_cust_trans_data['hour'] >=3) & (Cleaned_cust_trans_data['hour'] <=8)).astype(int)
Cleaned_cust_trans_data['amount_high'] = (Cleaned_cust_trans_data['amount_usd'] >500).astype(int)
Cleaned_cust_trans_data['high_ip_risk'] = (Cleaned_cust_trans_data['ip_risk_score'] >0.8).astype(int)
Cleaned_cust_trans_data['low_device_trust'] = (Cleaned_cust_trans_data['device_trust_score'] <0.5).astype(int)
Cleaned_cust_trans_data['new_account'] = ((Cleaned_cust_trans_data['account_age_days'] >=30) & (Cleaned_cust_trans_data['account_age_days'] <90)).astype(int)
Cleaned_cust_trans_data['very_new_account'] = (Cleaned_cust_trans_data['account_age_days'] <30).astype(int)
Cleaned_cust_trans_data['velocity_spike'] = (Cleaned_cust_trans_data['txn_velocity_1h'] >=3).astype(int)

high_fraud_tendency_signal_features = Cleaned_cust_trans_data[['late_night_hours', 'amount_high', 'high_ip_risk', 'low_device_trust', 'new_account', 'very_new_account', 'velocity_spike']]
high_fraud_tendency_signal_features.head(10)

,late_night_hours,amount_high,high_ip_risk,low_device_trust,new_account,very_new_account,velocity_spike
0,0,0,0,0,0,0,0
1,0,0,0,1,0,0,0
2,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0
5,0,1,0,0,0,0,0
6,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0


In [5]:
high_fraud_tendency_signal_features.columns

Index(['late_night_hours', 'amount_high', 'high_ip_risk', 'low_device_trust',
       'new_account', 'very_new_account', 'velocity_spike'],
      dtype='object')

### Feature Selection

With the new threshold features in place, this section removes columns that shouldn't be fed to a model — raw identifiers, the temporary bucket columns created only for EDA charting, and a small number of fields judged not useful or too risky to keep — before splitting what remains into categorical and numerical groups for preprocessing.

In [1]:
import sys
print(sys.executable)

c:\Users\oluwa\OneDrive\Desktop\Fraudulent_transaction_Detection_for_a_FinTech_Company\finlora_env\Scripts\python.exe


In [13]:
Cleaned_cust_trans_data.shape

(10650, 42)

In [ ]:

Cleaned_cust_trans_data.columns

Index(['Unnamed: 0', 'transaction_id', 'customer_id', 'timestamp',
       'home_country', 'source_currency', 'dest_currency', 'channel',
       'amount_src', 'amount_usd', 'fee', 'exchange_rate_src_to_dest',
       'device_id', 'new_device', 'ip_address', 'ip_country',
       'location_mismatch', 'ip_risk_score', 'kyc_tier', 'account_age_days',
       'device_trust_score', 'chargeback_history_count', 'risk_score_internal',
       'txn_velocity_1h', 'txn_velocity_24h', 'corridor_risk', 'is_fraud',
       'device_trust_score_bucket', 'ip_risk_score_bucket',
       'amount_usd_bucket', 'hour', 'day_of_week', 'is_weekend', 'month',
       'account_age_bucket', 'late_night_hours', 'amount_high', 'high_ip_risk',
       'low_device_trust', 'new_account', 'very_new_account',
       'velocity_spike'],
      dtype='object')

Listing every column currently in the dataset before deciding what to drop — this is the full inventory the next cell's three drop operations are applied against.

In [ ]:
# Dropping all identifier columns

Cleaned_cust_trans_data = Cleaned_cust_trans_data.drop(['transaction_id', 'customer_id', 'device_id', 'ip_address'], axis=1)

# Dropping all temporary bucket columns

Cleaned_cust_trans_data = Cleaned_cust_trans_data.drop(['device_trust_score_bucket', 'ip_risk_score_bucket', 'amount_usd_bucket', 'account_age_bucket'], axis=1)

# Dropping irrelevant variables

Cleaned_cust_trans_data = Cleaned_cust_trans_data.drop(['Unnamed: 0', 'exchange_rate_src_to_dest', 'chargeback_history_count', 'month'], axis=1)

**Three groups of columns are removed here, for three different reasons:**

1. **Identifiers** (transaction_id, customer_id, device_id, ip_address) — unique or near-unique per row, so they carry no pattern a model could generalise from to a new transaction or customer.
2. **Temporary bucket columns** (device_trust_score_bucket, ip_risk_score_bucket, amount_usd_bucket, account_age_bucket) — these existed only to make EDA charts readable; the underlying continuous values (device_trust_score, ip_risk_score, amount_usd, account_age_days) are still present and are what the model will actually use, alongside the new threshold flags engineered above.
3. **Unnamed: 0, exchange_rate_src_to_dest, chargeback_history_count, month** — a mixed group. Unnamed: 0 is a leftover CSV index column carrying no information; month is redundant now that hour/day_of_week/is_weekend exist. chargeback_history_count is dropped here as a standard precaution against target leakage. Because it's plausible that a chargeback could be logged as a direct consequence of a transaction already being identified as fraudulent, including it risks letting the model effectively see the outcome it's meant to predict, rather than a genuine leading indicator. Removing it keeps the feature set honest to what would actually be available at the moment of prediction in production.

In [20]:
Cleaned_cust_trans_data.shape

(10650, 30)

In [22]:
Cleaned_cust_trans_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10650 entries, 0 to 10649
Data columns (total 30 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   timestamp            10650 non-null  datetime64[ns, UTC]
 1   home_country         10650 non-null  object             
 2   source_currency      10650 non-null  object             
 3   dest_currency        10650 non-null  object             
 4   channel              10650 non-null  object             
 5   amount_src           10650 non-null  float64            
 6   amount_usd           10650 non-null  float64            
 7   fee                  10650 non-null  float64            
 8   new_device           10650 non-null  bool               
 9   ip_country           10650 non-null  object             
 10  location_mismatch    10650 non-null  bool               
 11  ip_risk_score        10650 non-null  float64            
 12  kyc_tier          

### Defining Categorical and Numerical Features



In [23]:
categorical_features = Cleaned_cust_trans_data.select_dtypes(include=['object', 'bool']).columns
categorical_features

Index(['home_country', 'source_currency', 'dest_currency', 'channel',
       'new_device', 'ip_country', 'location_mismatch', 'kyc_tier'],
      dtype='object')

In [24]:
numerical_features = Cleaned_cust_trans_data.select_dtypes(include=['int', 'float']).columns
numerical_features

Index(['amount_src', 'amount_usd', 'fee', 'ip_risk_score', 'account_age_days',
       'device_trust_score', 'risk_score_internal', 'txn_velocity_1h',
       'txn_velocity_24h', 'corridor_risk', 'is_fraud', 'hour', 'day_of_week',
       'is_weekend', 'late_night_hours', 'amount_high', 'high_ip_risk',
       'low_device_trust', 'new_account', 'very_new_account',
       'velocity_spike'],
      dtype='object')

In [25]:
print(f"Categorical: {len(categorical_features)}")
print(f"Numerical: {len(numerical_features)}")
print(f"Dataset: {Cleaned_cust_trans_data.shape}")

Categorical: 8
Numerical: 21
Dataset: (10650, 30)


8 categorical + 21 numerical = 29, one short of the 30 total columns — the missing one is timestamp , which doesn't match either dtype filter. Once is_fraud is correctly excluded from the numerical list , the real predictor count becomes 8 categorical + 20 numerical = 28 features, with timestamp (unused, its signal already captured elsewhere) and is_fraud (the target, correctly excluded from the feature lists) accounting for the remaining 2 of the 30 total columns.

In [26]:
Cleaned_cust_trans_data.to_csv(r'../Finlora_Dataset/artifacts/Engineered_Data.csv', index=False)

### Summary

This notebook converted the patterns identified during EDA into a modelling-ready feature set, and made several explicit inclusion/exclusion decisions ahead of preprocessing.

1. **Seven threshold-based features were engineered** (late_night_hours, high_ip_risk, low_device_trust, new_account, very_new_account, velocity_spike, amount_high), each directly traceable to a specific step-function relationship found in the EDA notebook — turning smooth or hard-to-learn continuous relationships into explicit binary signals a model can use immediately. 

2. **Twelve columns were dropped** across three groups: raw identifiers with no generalisable signal, temporary bucket columns superseded by their underlying continuous values, and a mixed group including chargeback_history_count — dropped as a precaution against potential target leakage that may be worth verifying against the source data dictionary.

3. **Categorical (8) and numerical (21) feature lists were defined**, with one open item flagged for correction before they're used to build the preprocessing pipeline: is_fraud is currently included in the numerical list and must be excluded to avoid handing the model its own answer as an input.

4. **The result is exported to Engineered_Data.csv**, ready for the modelling notebook.